In [1]:
#import the necessary packages for all analysis
import pandas as pd
import numpy as np 

In [2]:
#Import the data from Kaggle 
test_df = pd.read_csv('/Users/alextoy/Desktop/Kaggle Competitions /Titanic/titanic/test.csv')
train_df = pd.read_csv('/Users/alextoy/Desktop/Kaggle Competitions /Titanic/titanic/train.csv')

# Titanic Machine Learning - Gradient Boosting

## We'll use XGboost for this 

In [12]:
#now import the sklearn packages for splitting the sample and for running the Logit analysis 
#for splitting test/train and for k-folds cross validation 
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV, RandomizedSearchCV 

#to make transforming the data easier ColumnTransformer and Pipeline keep preprocessed data inside the folds 
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

#for encoding the data 
from sklearn.impute import SimpleImputer 
from sklearn.preprocessing import OneHotEncoder, StandardScaler

#to actually run the model 
#from sklearn.linear_model import LogisticRegression 
#from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

#to test how well the model performs 
from sklearn.metrics import log_loss, roc_auc_score, accuracy_score 

#to see which features are most important 
from sklearn.inspection import permutation_importance

In [5]:
#first create additional features
def add_titanic_features(df: pd.DataFrame) -> pd.DataFrame: 
    df = df.copy()

    #add log(fare) 
    df["log_Fare"] = np.log1p(df["Fare"])

    #family features (total size and if they're alone) 
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

    #title from name 
    df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.", expand = False).str.strip()
    df["Title"] = df["Title"].replace({
        "Mlle": "Miss", "Ms":"Miss","Mme":"Mrs"
    })
    df["Title"] = df["Title"].where(df["Title"].isin(["Mr","Mrs","Miss","Master"]), "Other")

    #Cabin Features 
    df["CabinKnown"] = df["Cabin"].notna().astype(int)
    df["Deck"] = df["Cabin"].str[0].fillna("Unknown")

    return df

In [6]:
#now add the additional covariates to the training data 
train_fe = add_titanic_features(train_df)

#define the feature list, then the X and y 
Feature_List = [
    "Pclass", "Sex", "Age",
    "log_Fare", "Embarked",
    "FamilySize", "IsAlone",
    "Title",
    "CabinKnown", "Deck",
]

X = train_fe[Feature_List]
y = train_fe["Survived"]

In [7]:
#now build the preprocessing 
numeric_features = ["Age","log_Fare","FamilySize"]
categorical_features = ["Pclass", "Sex", "Embarked",  "Title", "Deck"]
binary_features = ["IsAlone","CabinKnown"]
#numeric preprocessing
numeric_transformer = Pipeline(steps = [("imputer", SimpleImputer(strategy  = "median", add_indicator = True)),
                                         ]) # don't need scaling for RF

#categorical preprocessing 
categorical_transformer = Pipeline(steps = [("imputer", SimpleImputer(strategy = "constant", fill_value = "Missing")),
                                             ("onehot", OneHotEncoder(handle_unknown = "ignore"))]) # don't need to drop for RF

#binary preprocessing 
binary_transformer = Pipeline(steps = [("imputer", SimpleImputer(strategy = "most_frequent", add_indicator = True))])
#preprocess 
preprocess = ColumnTransformer(transformers = [("num", numeric_transformer, numeric_features),
                                              ("cat",categorical_transformer, categorical_features),
                                              ("bin", binary_transformer, binary_features)])


## XGBoost Simple 80-20 Split

In [8]:
#split the sample
X_train, X_val, y_train, y_val = train_test_split(X,y, test_size = 0.2, random_state = 1234, stratify = y)

#define the XGB model 
xgb_base = XGBClassifier(n_estimators = 500,
                                 random_state = 1234, 
                                 n_jobs = -1,
                                 learning_rate =0.5,
                                 max_depth = 3,
                                 subsample = 0.8,
                                 colsample_bytree = 0.8,
                                 reg_lambda = 1.0,
                                 eval_metric = "logloss")

xgb_pipe = Pipeline(steps = [("preprocess", preprocess),
                                 ("model", xgb_base)])

#fit the model 
xgb_pipe.fit(X_train, y_train)

#Evaluate the model
probability_xgb = xgb_pipe.predict_proba(X_val)[:,1]
prediction_xgb = (probability_xgb > 0.5).astype(int)

#RF Evaluation 
xgb_eval = {
    "log_loss": log_loss(y_val, probability_xgb),
    "roc_auc": roc_auc_score(y_val, probability_xgb),
    "accuracy": accuracy_score(y_val, prediction_xgb)
}
print("xgb eval:" ,xgb_eval)
print("Logit: {'log loss': 0.45475794869118946, 'ROC-AUC Score': 0.8483530961791831, 'Accuracy': 0.8156424581005587}")

xgb eval: {'log_loss': 0.7766253493743956, 'roc_auc': 0.8205533596837945, 'accuracy': 0.7988826815642458}
Logit: {'log loss': 0.45475794869118946, 'ROC-AUC Score': 0.8483530961791831, 'Accuracy': 0.8156424581005587}


## Now hypertune the XGB parameters for the 80-20 split 

In [9]:
inner_cv = StratifiedKFold(n_splits = 5, random_state = 1234, shuffle = True)

#define the tuned xgb model
xgb_base_tune = XGBClassifier(random_state = 1234, 
                                 n_jobs = -1,
                                 eval_metric = "logloss")

xgb_pipe_tune = Pipeline(steps = [("preprocess", preprocess),
                                 ("model", xgb_base_tune)])

param_grid = param_grid = {
    "model__n_estimators": [300, 600, 900],
    "model__learning_rate": [0.03, 0.05, 0.1],
    "model__max_depth": [2, 3, 5],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0],
    "model__reg_lambda": [1.0, 5.0],         # L2 regularization
    "model__min_child_weight": [1, 5],        # leaf “min data” control (regularization)
}

#now gridsearch using the inner_cv
xgb_search = GridSearchCV(
    xgb_pipe_tune,
    param_grid=param_grid,
    scoring="neg_log_loss",   # or "roc_auc" if ranking is your priority
    cv=inner_cv,
    n_jobs=-1
)

xgb_search.fit(X_train, y_train)
best_xgb = xgb_search.best_estimator_

print("Best params:", xgb_search.best_params_)
print("Best CV log loss (train folds):", -xgb_search.best_score_)

# Evaluate on holdout
probability_xgb_tune = best_xgb.predict_proba(X_val)[:, 1]
prediction_xgb_tune  = (probability_xgb_tune >= 0.5).astype(int)

xgb_eval_best_tune = {
    "log_loss": log_loss(y_val, probability_xgb_tune),
    "roc_auc": roc_auc_score(y_val, probability_xgb_tune),
    "accuracy": accuracy_score(y_val, prediction_xgb_tune)
}
print("XGB (tuned) holdout:", xgb_eval_best_tune)
print("Logit: {'log loss': 0.45475794869118946, 'ROC-AUC Score': 0.8483530961791831, 'Accuracy': 0.8156424581005587}")

Best params: {'model__colsample_bytree': 1.0, 'model__learning_rate': 0.03, 'model__max_depth': 3, 'model__min_child_weight': 1, 'model__n_estimators': 300, 'model__reg_lambda': 1.0, 'model__subsample': 0.8}
Best CV log loss (train folds): 0.4003920075725361
XGB (tuned) holdout: {'log_loss': 0.453220299149485, 'roc_auc': 0.8243083003952569, 'accuracy': 0.8212290502793296}
Logit: {'log loss': 0.45475794869118946, 'ROC-AUC Score': 0.8483530961791831, 'Accuracy': 0.8156424581005587}


## Now do stratified k-fold with inner tuning. 

In [10]:
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1234)
inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1234)

xgb_nested = GridSearchCV(
    xgb_pipe_tune,
    param_grid=param_grid,
    scoring="neg_log_loss",
    cv=inner_cv,
    n_jobs=-1
)

scoring = {"logloss": "neg_log_loss", "roc_auc": "roc_auc", "accuracy": "accuracy"}

nested = cross_validate(
    xgb_nested, X, y,
    cv=outer_cv,
    scoring=scoring,
    return_train_score=False
)

print("Nested XGB 5-fold CV")
print("Log loss:", (-nested["test_logloss"]).mean(), "±", (-nested["test_logloss"]).std())
print("ROC-AUC: ", (nested["test_roc_auc"]).mean(), "±", (nested["test_roc_auc"]).std())
print("Accuracy:", (nested["test_accuracy"]).mean(), "±", (nested["test_accuracy"]).std())
print("Logit: {'log loss': 0.45475794869118946, 'ROC-AUC Score': 0.8483530961791831, 'Accuracy': 0.8156424581005587}")

Nested XGB 5-fold CV
Log loss: 0.4054969808786956 ± 0.029180831997583603
ROC-AUC:  0.8735843204090934 ± 0.023270146391903165
Accuracy: 0.8226853304877284 ± 0.01746407410820913


## Now use the XGBoost algorithm with a finer parameter grid and the entire dataset for kaggle submission

In [17]:
inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1234)

#finer parameter grid 
param_distributions = {
    "model__n_estimators": np.arange(200, 1400, 100),
    "model__learning_rate": np.array([0.01, 0.02, 0.03, 0.05, 0.1]),
    "model__max_depth": np.arange(2, 7),              # 2..6
    "model__subsample": np.array([0.6, 0.7, 0.8, 0.9, 1.0]),
    "model__colsample_bytree": np.array([0.6, 0.7, 0.8, 0.9, 1.0]),
    "model__min_child_weight": np.array([1, 2, 5, 10]),
    "model__reg_lambda": np.array([0.5, 1.0, 2.0, 5.0, 10.0]),
    # optional:
    "model__gamma": np.array([0.0, 0.1, 0.3, 0.5]),
    "model__reg_alpha": np.array([0.0, 0.1, 0.5, 1.0]),
}

xgb_nested = RandomizedSearchCV(
    xgb_pipe_tune,
    param_distributions=param_distributions,
    scoring="accuracy",
    n_iter = 2000,
    cv=inner_cv,
    n_jobs=-1
)

xgb_nested.fit(X,y)

print("Best CV accuracy:", xgb_nested.best_score_)
print("Best params:", xgb_nested.best_params_)

best_model = xgb_nested.best_estimator_

Best CV accuracy: 0.8462243424769318
Best params: {'model__subsample': np.float64(0.8), 'model__reg_lambda': np.float64(2.0), 'model__reg_alpha': np.float64(0.5), 'model__n_estimators': np.int64(800), 'model__min_child_weight': np.int64(10), 'model__max_depth': np.int64(6), 'model__learning_rate': np.float64(0.05), 'model__gamma': np.float64(0.1), 'model__colsample_bytree': np.float64(1.0)}


In [24]:
X_test = add_titanic_features(test_df)
X_test = X_test[Feature_List]
test_proba = best_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= 0.85).astype(int)

print("Train columns match test columns:",
      list(X.columns) == list(X_test.columns))

submission = pd.DataFrame({
    "PassengerId": test_df["PassengerId"],
    "Survived": test_pred
})

submission.to_csv("/Users/alextoy/Desktop/Kaggle Competitions /Titanic/titanic/submission_03012026.csv", index=False)
submission.head()

Train columns match test columns: True


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,0
